# 82 Judge: choose the margin, score, report (CPU runtime)

1. **Choose the margin on validation.** For each candidate set: best = the highest p; shortlist = the candidates with p at least best minus delta; answer = the shortlisted candidate with the lowest predicted cost. Delta is chosen on validation only (the smallest cost among the deltas whose EX is not significantly below the best). Dev and mini-dev are then answered with that delta. Three answer files are written: the judge (`judge_v1_sel`), the judge with delta 0, which ignores cost (`judge_v1_sel_p`), and the most frequent candidate without any judge (`judge_v1_sel_vote`).
2. **Score** the three on validation, dev and mini-dev with the usual evaluation.
3. **Report**: everything against the generator alone (`rl_v1_it8`), with the reference models (OmniSQL-7B, Arctic-R1-7B, Qwen-32B).

**Decision gate:** if the judge does not beat the generator's single answer on validation (table 1 and the val part of table 3), we stop here and write up.

**Runtime > Change runtime type > CPU.** About 45 minutes, resumable.

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
DATA = "judge_data_v1"    # the judge's training data (built from own_v1 and pool_v1, already on Drive)
JUDGE = "judge_v1"        # the judge: adapter + cost head
CAND = "judge_cand_v1"    # candidates, scores and labels for val, dev and mini-dev
SEL = "judge_v1_sel"      # the selected answers (also judge_v1_sel_p and judge_v1_sel_vote)
GEN = "rl_v1_it8"         # the generator
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])

### 1. Choose delta on validation; write the selected answers

The first table (validation) decides; the dev and mini-dev tables are shown for the analysis only. Oracle: some candidate is correct (the ceiling of any selection).

In [ ]:
!python -m effsql.judge select --run {CAND} --judge {JUDGE} --out {SEL}

### 2. Stage the databases and score the selected answers (resumable)

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
for r, method in ((SEL, "judge"), (f"{SEL}_p", "judge-p"), (f"{SEL}_vote", "vote")):
    !python -m effsql.train eval-val --run {r}
    !effsql evaluate --pred {R}/{r}/predictions_dev.jsonl --split dev --timing none --run {r}_dev --model {MODEL} --method {method}
    !effsql evaluate --pred {R}/{r}/predictions_minidev.jsonl --split minidev --timing quick --run {r}_minidev --model {MODEL} --method {method}

### 3. Report: everything against the generator alone

Columns as in the earlier reports: EX, R-VES (step-based, gold = 100, wrong = 0) with a 95% bootstrap interval, R-VES/EX, answers at least 2x cheaper or slower than gold, geometric mean cost relative to gold. Below each table, each model against the generator on the same questions (EX and R-VES differences with p-values; cheaper / costlier when both are correct, by at least 10%). Saved as `judge_v1_report/report.txt`.

In [ ]:
RUNS = ["pool_v1_{split}_base", GEN, f"{SEL}_vote", f"{SEL}_p", SEL,
        "ref_omnisql7b", "ref_arctic7b", "ref_qwen32b"]
LABELS = ",".join(["pool_v1_{split}_base=zero-shot 7B", f"{GEN}=generator alone (RL)",
                   f"{SEL}_vote=text majority", f"{SEL}_p=judge, correctness only", f"{SEL}=judge, cost-aware",
                   "ref_omnisql7b=OmniSQL-7B", "ref_arctic7b=Arctic-R1-7B", "ref_qwen32b=Qwen-32B zero-shot"])
!python -m effsql.train report --runs "{','.join(RUNS)}" --ref {GEN} --labels "{LABELS}" --out judge_v1_report

### Done: disconnect the runtime and tell me the numbers are in

I read `judge_v1_sel/selection.txt` and `judge_v1_report/report.txt` from Drive and we decide on the ablations (notebook 83).